# Chuẩn bị bundle MIL trước khi huấn luyện

Notebook này chỉ chạy các lệnh kiểm kê, kiểm chứng và đóng gói CPU của `histology_data.pretrain_cli`. Notebook không huấn luyện mô hình, không tạo hoặc dừng VM, không cắt lại patch, không đổi màu và không mã hóa lại ảnh.

Mặc định chạy `draft`: tạo danh sách ca cần review từ Metadata và inventory ZIP. Sau khi điền đủ `patient_id`, xác minh danh tính/nhãn và bằng chứng theo từng ca, lưu file đã review tại `governance/reviews/case_review.csv`. Chỉ chuyển sang `build` sau khi có file review hợp lệ và feature release đã `complete` cùng full dataset audit. `common` là cohort 16 ca có đủ ba vật kính; `all` giữ đủ 18 ca với độ phủ vật kính khác nhau. `verify` kiểm tra bundle đã tạo.

Chạy trên Colab CPU. Nếu Drive đã mount trong phiên, notebook sẽ bỏ qua lệnh mount. Runtime bundle phải được xác thực bằng SHA-256 đã được người quản lý điền sau khi build.


In [ ]:
import importlib.util
import os
from pathlib import Path


def _safe_find_spec(name):
    try:
        return importlib.util.find_spec(name)
    except (ModuleNotFoundError, ValueError):
        return None

GOOGLE_PARENT_AVAILABLE = _safe_find_spec('google') is not None
GOOGLE_COLAB_AVAILABLE = (
    _safe_find_spec('google.colab') is not None if GOOGLE_PARENT_AVAILABLE else False
)
IS_COLAB = GOOGLE_PARENT_AVAILABLE and GOOGLE_COLAB_AVAILABLE
DRIVE_MOUNT = Path('/content/drive')
ROOT = Path('/content/drive/MyDrive/histology')

if IS_COLAB:
    from google.colab import drive

    DRIVE_ALREADY_AVAILABLE = (
        os.path.ismount(str(DRIVE_MOUNT)) or (DRIVE_MOUNT / 'MyDrive').is_dir()
    )
    if not DRIVE_ALREADY_AVAILABLE:
        drive.mount(str(DRIVE_MOUNT))

print('Environment:', 'Colab' if IS_COLAB else 'local / Drive must already be available')
print('Histology root:', ROOT)


## Cấu hình đầu vào

Giữ `RUN_MODE='draft'` cho lượt đầu. Runtime nằm riêng dưới `/content`, còn ZIP, metadata, feature và bundle dùng Drive. Các số kỳ vọng bên dưới mô tả cohort đã xác nhận; đặt một giá trị về `None` nếu tái sử dụng notebook cho cohort khác. Không đặt `RUN_MODE='build'` cho đến khi review và feature audit đều hoàn tất.


In [ ]:
from pathlib import Path

# Run modes: draft, build, verify.
RUN_MODE = 'draft'
SOURCE_KIND = 'zip'  # 'zip' or 'directory'.
COHORT = 'common'  # 'common' (16 three-lens cases) or 'all' (18 cases).

# ROOT is set in the environment cell; keep every path derived from that one value.
METADATA = ROOT / 'source' / 'Metadata.xlsx'
ZIP_SOURCE_ROOT = ROOT / 'source' / 'archives'
DIRECTORY_SOURCE_ROOT = ROOT / 'source' / 'tiles'
SOURCE_ROOT = ZIP_SOURCE_ROOT if SOURCE_KIND == 'zip' else DIRECTORY_SOURCE_ROOT

RUNTIME_ZIP = ROOT / 'runtime' / 'histology-pretrain-runtime.zip'
EXPECTED_RUNTIME_SHA256 = 'be5e62b12018b2cdace3020e81ed2537c9595e7112a49f47121ba18546d0db87'
RUNTIME_ROOT = (Path('/content/histology-pretrain-runtime') if IS_COLAB
                else ROOT / '.runtime' / 'histology-pretrain-runtime')
ENCODER_ROOT = ROOT / 'encoders'  # Read-only for this notebook.

FEATURE_ROOT = ROOT / 'features' / 'v001'
CASE_REVIEW = ROOT / 'governance' / 'reviews' / 'case_review.csv'
DUPLICATE_DISPOSITIONS = ROOT / 'governance' / 'reviews' / 'duplicate_dispositions.csv'
DRAFT_OUTPUT_ROOT = ROOT  # Writes ROOT/governance/drafts/<draft_id>/.
BUNDLE_OUTPUT_ROOT = ROOT  # Builder writes ROOT/bundles/<bundle_id>/.
BUNDLE_PATH = None  # Set to the bundle directory before RUN_MODE='verify'.

EXPECTED_SOURCES = 25
EXPECTED_PATCHES = 148991
EXPECTED_VECTORS = 148991
EXPECTED_CASES = 18
EXPECTED_PATIENTS = 18
EXPECTED_LABEL_COUNTS = {'0': 10, '1': 8}

if RUN_MODE not in {'draft', 'build', 'verify'}:
    raise ValueError("RUN_MODE must be 'draft', 'build', or 'verify'.")
if SOURCE_KIND not in {'zip', 'directory'}:
    raise ValueError("SOURCE_KIND must be 'zip' or 'directory'.")
if COHORT not in {'common', 'all'}:
    raise ValueError("COHORT must be 'common' or 'all'.")


## Tải runtime an toàn và kiểm tra đầu vào

Runtime phải là bundle được duyệt, có SHA-256 64 ký tự. Notebook từ chối placeholder, đường dẫn ZIP nguy hiểm, symlink và tổng nội dung giải nén trên 2 MiB. Runtime được giải nén vào thư mục tạm riêng; notebook chỉ sửa các package còn thiếu, không nâng phiên bản PyTorch, torchvision hoặc Pillow đang có.


In [ ]:
import csv
import hashlib
import importlib.util as importlib_util
import json
import os
import re
import shutil
import stat
import subprocess
import sys
import tempfile
import zipfile
from pathlib import Path, PurePosixPath, PureWindowsPath

MAX_RUNTIME_UNCOMPRESSED_BYTES = 2 * 1024 * 1024
CASE_REVIEW_FIELDS = (
    'case_id', 'patient_id', 'identity_verified', 'identity_evidence', 'case_label',
    'label_verified', 'label_evidence', 'raw_glade_values_json',
    'raw_conclusion_values_json', 'candidate_label_codes_json', 'metadata_case_sha256',
)

def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def _safe_runtime_relative_path(info):
    raw_name = info.filename
    name = raw_name[:-1] if info.is_dir() and raw_name.endswith('/') else raw_name
    if not name or '\x00' in name or '\\' in name:
        raise ValueError('Runtime ZIP contains an unsafe member path.')
    posix_path = PurePosixPath(name)
    windows_path = PureWindowsPath(name)
    parts = name.split('/')
    if (posix_path.is_absolute() or windows_path.is_absolute() or windows_path.drive
            or ':' in name or any(part in {'', '.', '..'} for part in parts)
            or posix_path.as_posix() != name):
        raise ValueError('Runtime ZIP contains an unsafe member path.')
    mode = info.external_attr >> 16
    if stat.S_ISLNK(mode):
        raise ValueError('Runtime ZIP contains a symbolic link.')
    mode_type = stat.S_IFMT(mode)
    if mode_type not in {0, stat.S_IFREG, stat.S_IFDIR}:
        raise ValueError('Runtime ZIP contains a non-regular member.')
    return posix_path

def validate_and_extract_runtime(runtime_zip, expected_sha256, runtime_root, encoder_root):
    runtime_zip = Path(runtime_zip)
    runtime_root = Path(runtime_root)
    encoder_root = Path(encoder_root)
    expected_sha256 = str(expected_sha256).strip().lower()
    if not re.fullmatch(r'[0-9a-f]{64}', expected_sha256):
        raise ValueError('EXPECTED_RUNTIME_SHA256 must be a reviewed 64-character SHA-256.')
    if not runtime_zip.is_file():
        raise FileNotFoundError('The external pretrain runtime ZIP is missing.')
    actual_sha256 = _sha256_file(runtime_zip)
    if actual_sha256 != expected_sha256:
        raise ValueError('Runtime ZIP SHA-256 does not match the reviewed value.')
    runtime_path = runtime_root.resolve()
    encoder_path = encoder_root.resolve()
    if (runtime_path == encoder_path or runtime_path.is_relative_to(encoder_path)
            or encoder_path.is_relative_to(runtime_path)):
        raise ValueError('Runtime extraction must stay separate from the encoder directory.')

    with zipfile.ZipFile(runtime_zip) as archive:
        files = [info for info in archive.infolist() if not info.is_dir()]
        declared_size = sum(info.file_size for info in files)
        if any(info.file_size < 0 for info in files) or declared_size > MAX_RUNTIME_UNCOMPRESSED_BYTES:
            raise ValueError('Runtime ZIP exceeds the 2 MiB uncompressed code cap.')
        members = []
        seen = set()
        for info in archive.infolist():
            relative = _safe_runtime_relative_path(info)
            key = relative.as_posix()
            if key in seen:
                raise ValueError('Runtime ZIP contains duplicate member paths.')
            seen.add(key)
            members.append((info, relative))

        runtime_root.parent.mkdir(parents=True, exist_ok=True)
        staging_root = Path(tempfile.mkdtemp(
            prefix=runtime_root.name + '.staging-', dir=runtime_root.parent
        ))
        try:
            written_total = 0
            for info, relative in members:
                destination = staging_root.joinpath(*relative.parts)
                if not destination.resolve().is_relative_to(staging_root.resolve()):
                    raise ValueError('Runtime member escapes its staging directory.')
                if info.is_dir():
                    destination.mkdir(parents=True, exist_ok=True)
                    continue
                destination.parent.mkdir(parents=True, exist_ok=True)
                member_bytes = 0
                with archive.open(info, 'r') as source, destination.open('wb') as target:
                    while True:
                        chunk = source.read(64 * 1024)
                        if not chunk:
                            break
                        member_bytes += len(chunk)
                        written_total += len(chunk)
                        if member_bytes > info.file_size or written_total > MAX_RUNTIME_UNCOMPRESSED_BYTES:
                            raise ValueError('Runtime ZIP exceeds its declared size or 2 MiB cap.')
                        target.write(chunk)
                if member_bytes != info.file_size:
                    raise ValueError('Runtime ZIP member size does not match its directory record.')
            required = (
                staging_root / 'histology_data' / '__init__.py',
                staging_root / 'histology_data' / 'pretrain_cli.py',
                staging_root / 'requirements-pretrain.txt',
            )
            if not all(path.is_file() for path in required):
                raise RuntimeError('Runtime ZIP is missing histology_data.pretrain_cli or requirements-pretrain.txt.')
            if runtime_root.exists():
                if runtime_root.is_symlink() or not runtime_root.is_dir():
                    raise ValueError('Runtime extraction target is not a regular directory.')
                shutil.rmtree(runtime_root)
            staging_root.replace(runtime_root)
        except Exception:
            shutil.rmtree(staging_root, ignore_errors=True)
            raise
    return {'sha256': actual_sha256, 'runtime_root': str(runtime_root),
            'uncompressed_bytes': declared_size}

REQUIREMENT_IMPORTS = {
    'numpy': 'numpy', 'pandas': 'pandas', 'pillow': 'PIL', 'openpyxl': 'openpyxl',
    'opencv-python-headless': 'cv2', 'scikit-learn': 'sklearn',
}

def _runtime_requirement_specs(requirements_path, runtime_root, seen=None):
    requirements_path = Path(requirements_path)
    runtime_root = Path(runtime_root).resolve()
    seen = set() if seen is None else seen
    resolved = requirements_path.resolve()
    if not resolved.is_relative_to(runtime_root):
        raise ValueError('Runtime requirement include escapes the bundle.')
    if resolved in seen:
        return []
    seen.add(resolved)
    if not resolved.is_file():
        raise FileNotFoundError('A runtime requirements file is missing.')
    specs = []
    for line in resolved.read_text(encoding='utf-8').splitlines():
        item = line.strip()
        if not item or item.startswith('#'):
            continue
        if item.startswith(('-r ', '--requirement ')):
            included = item.split(None, 1)[1]
            specs.extend(_runtime_requirement_specs(resolved.parent / included, runtime_root, seen))
            continue
        if item.startswith('-'):
            raise ValueError('Unsupported option in runtime requirements.')
        match = re.match(r'([A-Za-z0-9_.-]+)', item)
        if not match:
            raise ValueError('Unsupported runtime requirement format.')
        distribution = match.group(1).lower().replace('_', '-')
        if distribution in {'torch', 'torchvision'}:
            raise ValueError('The pretrain runtime must not install PyTorch or torchvision.')
        module = REQUIREMENT_IMPORTS.get(distribution, distribution.replace('-', '_'))
        specs.append((distribution, module, item))
    return specs

def install_missing_runtime_requirements(runtime_root):
    requirements = Path(runtime_root) / 'requirements-pretrain.txt'
    specs = _runtime_requirement_specs(requirements, runtime_root)
    missing_specs = []
    missing_names = []
    seen = set()
    for distribution, module, requirement in specs:
        if distribution in seen:
            continue
        seen.add(distribution)
        try:
            available = importlib_util.find_spec(module) is not None
        except (ImportError, ModuleNotFoundError, ValueError):
            available = False
        if not available:
            missing_specs.append(requirement)
            missing_names.append(distribution)
    if missing_specs:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check',
             '--no-deps', *missing_specs],
            check=True, capture_output=True, text=True,
        )
    return missing_names

def inspect_case_review_file(path, expected_cases=None, expected_patients=None,
                             expected_label_counts=None):
    path = Path(path)
    blockers = []
    counts = {'case_count': 0, 'unique_patient_count': 0, 'label_counts': {'0': 0, '1': 0}}
    if not path.is_file():
        return {'ready': False, 'blockers': ['case_review_missing'], **counts}
    try:
        with path.open(encoding='utf-8-sig', newline='') as stream:
            reader = csv.DictReader(stream)
            fields = tuple(reader.fieldnames or ())
            rows = list(reader)
    except (OSError, UnicodeError, csv.Error):
        return {'ready': False, 'blockers': ['case_review_unreadable'], **counts}
    if fields != CASE_REVIEW_FIELDS:
        return {'ready': False, 'blockers': ['case_review_schema_mismatch'], **counts}
    case_ids = [row.get('case_id', '').strip() for row in rows]
    patient_ids = [row.get('patient_id', '').strip() for row in rows]
    labels = [row.get('case_label', '').strip() for row in rows]
    counts['case_count'] = len(rows)
    counts['unique_patient_count'] = len({value for value in patient_ids if value})
    counts['label_counts'] = {label: labels.count(label) for label in ('0', '1')}
    if not rows:
        blockers.append('case_review_empty')
    if any(not value for value in case_ids) or len(case_ids) != len(set(case_ids)):
        blockers.append('case_ids_missing_or_duplicated')
    if any(not value for value in patient_ids):
        blockers.append('patient_ids_missing')
    if any(row.get('identity_verified', '').strip().casefold() != 'true'
           or not row.get('identity_evidence', '').strip() for row in rows):
        blockers.append('identity_review_or_evidence_incomplete')
    if any(label not in {'0', '1'} for label in labels):
        blockers.append('case_labels_not_binary')
    if any(row.get('label_verified', '').strip().casefold() != 'true'
           or not row.get('label_evidence', '').strip() for row in rows):
        blockers.append('label_review_or_evidence_incomplete')
    provenance_fields = ('raw_glade_values_json', 'raw_conclusion_values_json',
                         'candidate_label_codes_json', 'metadata_case_sha256')
    if any(not row.get(field, '').strip() for row in rows for field in provenance_fields):
        blockers.append('source_provenance_incomplete')
    if expected_cases is not None and len(rows) != expected_cases:
        blockers.append('case_count_mismatch')
    if expected_patients is not None and counts['unique_patient_count'] != expected_patients:
        blockers.append('patient_count_mismatch')
    if expected_label_counts is not None:
        expected = {str(key): int(value) for key, value in expected_label_counts.items()}
        if counts['label_counts'] != expected:
            blockers.append('label_count_mismatch')
    if set(labels) != {'0', '1'}:
        blockers.append('binary_classes_incomplete')
    blockers = sorted(set(blockers))
    return {'ready': not blockers, 'blockers': blockers, **counts}

def inspect_feature_release(feature_root, expected_vectors=None):
    feature_root = Path(feature_root)
    blockers = []
    result = {'ready': False, 'blockers': [], 'verified_vectors': None}
    status_path = feature_root / 'status.json'
    if not status_path.is_file():
        return {'ready': False, 'blockers': ['feature_status_missing'], 'verified_vectors': None}
    try:
        status = json.loads(status_path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return {'ready': False, 'blockers': ['feature_status_unreadable'], 'verified_vectors': None}
    feature_id = str(status.get('feature_id', ''))
    if not re.fullmatch(r'[0-9a-f]{64}', feature_id):
        blockers.append('feature_id_invalid')
        return {'ready': False, 'blockers': blockers, 'verified_vectors': None}
    release_root = feature_root / feature_id
    release_path = release_root / 'release.json'
    audit_path = release_root / 'dataset_audit.json'
    if status.get('status') != 'complete' or status.get('feature_complete') is not True:
        blockers.append('feature_status_incomplete')
    if not release_path.is_file():
        blockers.append('feature_release_marker_missing')
    if not audit_path.is_file():
        blockers.append('feature_dataset_audit_missing')
    if blockers:
        return {'ready': False, 'blockers': sorted(set(blockers)), 'verified_vectors': None}
    try:
        release = json.loads(release_path.read_text(encoding='utf-8'))
        audit = json.loads(audit_path.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return {'ready': False, 'blockers': ['feature_audit_unreadable'], 'verified_vectors': None}
    verified_vectors = audit.get('verified_vectors')
    result['verified_vectors'] = verified_vectors if isinstance(verified_vectors, int) else None
    if release.get('feature_id') != feature_id:
        blockers.append('feature_release_version_mismatch')
    if release.get('feature_complete') is not True or audit.get('feature_complete') is not True:
        blockers.append('feature_complete_marker_missing')
    if release.get('source_complete') is not True or audit.get('source_complete') is not True:
        blockers.append('feature_source_incomplete')
    if release.get('audit_complete') is not True:
        blockers.append('feature_release_audit_marker_missing')
    if audit.get('feature_id') != feature_id:
        blockers.append('feature_audit_version_mismatch')
    if expected_vectors is not None and verified_vectors != expected_vectors:
        blockers.append('verified_vector_count_mismatch')
    result['blockers'] = sorted(set(blockers))
    result['ready'] = not result['blockers']
    return result

def _latest_review_draft(root):
    draft_root = Path(root) / 'governance' / 'drafts'
    candidates = list(draft_root.glob('*/case_review.csv')) if draft_root.is_dir() else []
    if not candidates:
        return None
    return str(max(candidates, key=lambda path: path.stat().st_mtime).resolve())

def _safe_cli_summary(command, return_code, stdout, mode, output_root,
                      case_review_path=None, preflight=None):
    payload = {}
    try:
        parsed = json.loads(stdout)
        if isinstance(parsed, dict):
            payload = parsed
    except (TypeError, ValueError):
        pass
    detail_payload = payload.get('bundle') if mode == 'build' else payload
    if not isinstance(detail_payload, dict):
        detail_payload = payload
    raw_status = detail_payload.get('status')
    allowed_statuses = {
        'blocked', 'bundle_created', 'build_complete', 'complete', 'completed',
        'draft_blocked', 'draft_complete', 'draft_created', 'error', 'failed',
        'feature_incomplete', 'partial', 'ready', 'source_incomplete', 'verified',
    }
    cli_status = raw_status if isinstance(raw_status, str) and raw_status in allowed_statuses else None
    is_blocked = cli_status in {'blocked', 'draft_blocked', 'error', 'failed',
                                'feature_incomplete', 'source_incomplete'}
    result = {
        'mode': mode,
        'status': cli_status or ('completed' if return_code == 0 else 'error'),
        'return_code': int(return_code),
        'training_ready': (bool(detail_payload.get('training_ready'))
                           if return_code == 0 and mode in {'build', 'verify'}
                           and cli_status in {'ready', 'bundle_created', 'build_complete', 'complete',
                                              'completed', 'verified'} and not is_blocked else False),
        'output_root': str(Path(output_root)),
    }
    cli_blockers = detail_payload.get('blockers')
    if isinstance(cli_blockers, list):
        result['cli_blocker_count'] = len(cli_blockers)
    if mode == 'build':
        for source_key, output_key in (
            ('case_count', 'cohort_case_count'), ('patient_count', 'cohort_patient_count'),
            ('instance_count', 'selected_instance_count'), ('excluded_case_count', 'excluded_case_count'),
        ):
            value = detail_payload.get(source_key)
            if isinstance(value, int) and not isinstance(value, bool):
                result[output_key] = value
        if preflight:
            for source_key, output_key in (
                ('case_count', 'reviewed_case_count'), ('unique_patient_count', 'reviewed_patient_count'),
                ('verified_vectors', 'verified_vectors'),
            ):
                value = preflight.get(source_key)
                if isinstance(value, int):
                    result[output_key] = value
    else:
        for key in ('case_count', 'verified_vectors', 'observed_sources', 'expected_sources',
                    'observed_patches', 'expected_patches'):
            value = detail_payload.get(key)
            if isinstance(value, int) and not isinstance(value, bool):
                result[key] = value
        if preflight:
            for key in ('case_count', 'unique_patient_count', 'verified_vectors'):
                value = preflight.get(key)
                if isinstance(value, int):
                    result.setdefault(key, value)
    if mode == 'draft':
        result['draft_directory'] = str(Path(output_root) / 'governance' / 'drafts')
        result['review_csv_path'] = _latest_review_draft(output_root)
        result['official_review_path'] = str(case_review_path)
        result['review_instructions'] = (
            'Copy the draft CSV to the official review path; fill patient_id, identity evidence, '
            'binary case_label, and label evidence; set both verification fields true; preserve provenance columns.'
        )
    elif mode == 'build':
        result['cohort'] = COHORT
        candidate_path = detail_payload.get('bundle_path')
        if isinstance(candidate_path, str):
            try:
                Path(candidate_path).resolve().relative_to(Path(output_root).resolve())
            except (OSError, ValueError):
                pass
            else:
                result['bundle_path'] = candidate_path
    elif mode == 'verify':
        result['bundle_path'] = str(output_root)
    if return_code != 0:
        result['details'] = 'Raw CLI output is suppressed; inspect the command in a controlled session.'
    return result

def build_pretrain_command(mode):
    base = [sys.executable, '-m', 'histology_data.pretrain_cli', mode]
    if mode == 'draft':
        return base + [
            '--metadata', str(METADATA), '--source-root', str(SOURCE_ROOT),
            '--output-root', str(DRAFT_OUTPUT_ROOT), '--source-kind', SOURCE_KIND,
            '--expected-sources', str(EXPECTED_SOURCES),
            '--expected-patches', str(EXPECTED_PATCHES),
        ]
    if mode == 'build':
        command = base + [
            '--metadata', str(METADATA), '--case-review', str(CASE_REVIEW),
            '--feature-root', str(FEATURE_ROOT), '--output-root', str(BUNDLE_OUTPUT_ROOT),
            '--cohort', COHORT, '--expected-sources', str(EXPECTED_SOURCES),
            '--expected-vectors', str(EXPECTED_VECTORS),
        ]
        if DUPLICATE_DISPOSITIONS.is_file():
            command.extend(['--duplicate-dispositions', str(DUPLICATE_DISPOSITIONS)])
        return command
    if mode == 'verify':
        if BUNDLE_PATH is None:
            raise ValueError('Set BUNDLE_PATH to an existing bundle directory before verification.')
        return base + ['--bundle', str(BUNDLE_PATH), '--feature-root', str(FEATURE_ROOT)]
    raise ValueError('Unsupported pretrain command.')


## Chạy chế độ đã chọn

`build` kiểm tra review CSV theo đúng schema và bằng chứng từng hàng, rồi xác nhận feature status, release marker và dataset audit đủ 148.991 vector trước khi gọi CLI. CLI vẫn là nơi xác minh checksum, provenance và duplicate dispositions. Khi thiếu điều kiện, notebook trả `blocked` với `training_ready=false` và không gọi lệnh build.

Output chỉ gồm trạng thái, số lượng, đường dẫn cần thiết và hướng dẫn review. Notebook không in danh sách bệnh nhân/ca, nhãn từng ca, kết luận hoặc log thô từ CLI.


In [ ]:
def execute_readiness_mode():
    blockers = []
    preflight = {}
    if not ROOT.is_dir():
        blockers.append('histology_drive_root_missing')
    if RUN_MODE == 'draft':
        if not METADATA.is_file():
            blockers.append('metadata_missing')
        if not SOURCE_ROOT.is_dir():
            blockers.append('source_root_missing')
    elif RUN_MODE == 'build':
        review = inspect_case_review_file(
            CASE_REVIEW, expected_cases=EXPECTED_CASES,
            expected_patients=EXPECTED_PATIENTS,
            expected_label_counts=EXPECTED_LABEL_COUNTS,
        )
        feature = inspect_feature_release(FEATURE_ROOT, expected_vectors=EXPECTED_VECTORS)
        preflight = {
            'case_count': review['case_count'],
            'unique_patient_count': review['unique_patient_count'],
            'verified_vectors': feature['verified_vectors'],
        }
        blockers.extend(review['blockers'])
        blockers.extend(feature['blockers'])
    elif RUN_MODE == 'verify':
        if BUNDLE_PATH is None or not Path(BUNDLE_PATH).is_dir():
            blockers.append('bundle_path_missing')

    if blockers:
        status = {
            'mode': RUN_MODE, 'status': 'blocked', 'training_ready': False,
            'blockers': sorted(set(blockers)), 'output_root': str(
                DRAFT_OUTPUT_ROOT if RUN_MODE == 'draft' else BUNDLE_OUTPUT_ROOT
            ),
        }
        if RUN_MODE == 'draft':
            status.update({
                'official_review_path': str(CASE_REVIEW),
                'review_instructions': 'Resolve the listed input prerequisite, then rerun draft.',
            })
        elif RUN_MODE == 'build':
            status.update({
                'case_review_path': str(CASE_REVIEW),
                'feature_root': str(FEATURE_ROOT),
                'cohort': COHORT,
                'review_instructions': 'Complete the case review and full feature audit, then rerun build.',
            })
        elif RUN_MODE == 'verify' and BUNDLE_PATH is not None:
            status['bundle_path'] = str(BUNDLE_PATH)
        if preflight:
            status['preflight_counts'] = preflight
        return status

    runtime_info = validate_and_extract_runtime(
        RUNTIME_ZIP, EXPECTED_RUNTIME_SHA256, RUNTIME_ROOT, ENCODER_ROOT
    )
    installed = install_missing_runtime_requirements(RUNTIME_ROOT)
    command = build_pretrain_command(RUN_MODE)
    environment = os.environ.copy()
    old_pythonpath = environment.get('PYTHONPATH', '')
    environment['PYTHONPATH'] = str(RUNTIME_ROOT) + (os.pathsep + old_pythonpath if old_pythonpath else '')
    environment['CUDA_VISIBLE_DEVICES'] = ''
    completed = subprocess.run(
        command, cwd=RUNTIME_ROOT, env=environment,
        capture_output=True, text=True, check=False,
    )
    output_root = DRAFT_OUTPUT_ROOT if RUN_MODE == 'draft' else BUNDLE_OUTPUT_ROOT
    if RUN_MODE == 'verify':
        output_root = BUNDLE_PATH
    status = _safe_cli_summary(
        command, completed.returncode, completed.stdout, RUN_MODE, output_root,
        case_review_path=CASE_REVIEW, preflight=preflight,
    )
    status['runtime_sha256'] = runtime_info['sha256']
    status['runtime_root'] = runtime_info['runtime_root']
    status['missing_dependencies_installed'] = installed
    if RUN_MODE == 'draft' and completed.returncode == 0 and not status['review_csv_path']:
        status['status'] = 'error'
        status['training_ready'] = False
        status['details'] = 'CLI succeeded but no draft review CSV was found under the governance output root.'
    return status

try:
    RUN_STATUS = execute_readiness_mode()
except Exception as exc:
    RUN_STATUS = {
        'mode': RUN_MODE, 'status': 'error', 'training_ready': False,
        'error_type': type(exc).__name__,
        'details': 'Raw exceptions and CLI output are suppressed to avoid exposing case metadata.',
    }
print(json.dumps(RUN_STATUS, ensure_ascii=False, indent=2))


## Tiếp theo

Với trạng thái `draft`, mở file review CSV được in trong JSON output, xác nhận danh tính từng ca và điền nhãn nhị phân cùng nguồn bằng chứng. Giữ nguyên các cột provenance. Lưu bản đã review tại `governance/reviews/case_review.csv`.

Feature cache phải có release đầy đủ và `dataset_audit.json` xác nhận toàn bộ 148.991 vector. Sau đó chọn `RUN_MODE='build'` và `COHORT='common'` để đóng gói cohort 16 ca chung ba vật kính; chọn `all` nếu cần bundle 18 ca theo độ phủ riêng của từng vật kính. Chọn `verify` cùng `BUNDLE_PATH` để xác minh bundle.

Các lệnh này không train MIL. Notebook giữ mọi package code trong runtime tạm `/content/histology-pretrain-runtime`, không thay đổi encoder đã lưu và không dừng phiên Colab.
